In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

metadata_df = metadata_df[(metadata_df['dmfs time'] != 'NA') & (metadata_df['dmfs event'] != 'NA')]

metadata_df

,sample_title,gender,dmfs time,dmfs event,sample type,irdm subtype,pam50 subtype,immunity group,proliferation group,er status rna,pr status rna
samples,,,,,,,,,,,
GSM3121443,SH_TS_BC111,female,6.666666667,0,Solid tumor,Mixed,Mixed,iweak,high,ER+,PR+
GSM3121444,SH_TS_BC112,female,6.083333333,0,Solid tumor,Normal,Normal,iweak,low,ER+,PR+
GSM3121445,SH_TS_BC113,female,1.916666667,0,Solid tumor,Basal,Basal,iweak,high,ER-,PR+
GSM3121451,SH_TS_BC119,female,6.75,0,Solid tumor,HER2E,HER2E,istrong,high,ER-,PR+
GSM3121454,SH_TS_BC133,female,3.916666667,0,Solid tumor,Basal,Basal,iweak,high,ER+,PR-
...,...,...,...,...,...,...,...,...,...,...,...
GSM3121893,UNC_TGS_BC_96,female,5.33,0,Solid tumor,LumA,LumA,iweak,low,ER+,PR+
GSM3121894,UNC_TGS_BC_97,female,4.83,0,Solid tumor,LumA,LumA,iweak,low,ER+,PR+
GSM3121895,UNC_TGS_BC_99,female,5.42,0,Solid tumor,Basal,Basal,iweak,high,ER-,PR-


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_91679/3050319327.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,gender,dmfs time,dmfs event,sample type,irdm subtype,pam50 subtype,immunity group,proliferation group,er status rna,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM3121445,SH_TS_BC113,female,1.916666667,0,Solid tumor,Basal,Basal,iweak,high,ER-,...,0,0,0,0,0,0,0,0,0,0
GSM3121451,SH_TS_BC119,female,6.75,0,Solid tumor,HER2E,HER2E,istrong,high,ER-,...,0,0,0,0,0,0,0,0,0,0
GSM3121459,SH_TS_BC146,female,3.416666667,0,Solid tumor,Basal,Basal,iweak,high,ER+,...,0,0,0,0,0,0,0,0,0,0
GSM3121460,SH_TS_BC147,female,9.75,0,Solid tumor,LumB,LumB,iweak,high,ER+,...,0,0,0,0,0,0,0,0,0,0
GSM3121462,SH_TS_BC150,female,9.083333333,0,Solid tumor,LumA,LumA,istrong,low,ER+,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM3121893,UNC_TGS_BC_96,female,5.33,0,Solid tumor,LumA,LumA,iweak,low,ER+,...,0,0,0,0,0,0,0,0,0,0
GSM3121894,UNC_TGS_BC_97,female,4.83,0,Solid tumor,LumA,LumA,iweak,low,ER+,...,0,0,0,0,0,0,0,0,0,0
GSM3121895,UNC_TGS_BC_99,female,5.42,0,Solid tumor,Basal,Basal,iweak,high,ER-,...,0,0,0,0,1,1,0,0,0,0


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
